<a href="https://colab.research.google.com/github/ajeet0001/gpt-2-code-from-scratch/blob/main/GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### In this note book i code the complete gpt-2 model from scratch started from tokenization to implementing training loop and then loading openai weights and finally implementing temprature and top k techniques to control rnadomness of gpt-2 model

1. loading the book data

In [ ]:
import torch
import torch.nn as nn
import os
import urllib.request

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [ ]:
print(text_data[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [ ]:
print(text_data[-99:])

it for me! The Strouds stand alone, and happen once--but there's no exterminating our kind of art."


2. Tokenize the data

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

In [ ]:
total_char = len(text_data)
tokenize_char = len(tokenizer.encode(text_data))
print('total len of text data:',total_char)
print("len of data after tokenization:",tokenize_char)

total len of text data: 20479
len of data after tokenization: 5145


The data is very short to train an LLM but here we are using this for learning purpose and resource limitation

3. next we split the data into train and val set and use data loader to prepare batches for the training

In [ ]:
from torch.utils.data import Dataset, DataLoader
class GPTDatasetv1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride) -> None:
      super().__init__()
      self.input_ids = []
      self.target_ids = []
      # tokenize the entire text
      token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})
      # Use a sliding window to chunk the book into overlapping sequences of max_length
      for i in range(0,len(token_ids)-max_length,stride):
        input_chunk = token_ids[i:i+max_length]
        target_chunk = token_ids[i+1:i+max_length+1]
        self.input_ids.append(torch.tensor(input_chunk))
        self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
        return len(self.input_ids)

  def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
def create_dataloader_v1(
    txt, batch_size = 4, max_length = 256, stride = 128, shuffle = True, drop_last = True, num_workers = 0):
  # initialize the tokenizer
  tokenizer = tiktoken.get_encoding('gpt2')

  # create the dataset
  dataset = GPTDatasetv1(txt,tokenizer,max_length,stride)

  # create data loader
  dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
  return dataloader




In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 1024, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}


In [ ]:
# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# Sanity check

if tokenize_char * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if tokenize_char * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

Not enough tokens for the validation loader. Try to lower the `GPT_CONFIG_124M['context_length']` or decrease the `training_ratio`


In [ ]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

print(len(train_loader))
print(len(val_loader))


Train loader:
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])

Validation loader:
2
0


In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

Training tokens: 4096
Validation tokens: 0
All tokens: 4096


4. Implementing transformer block

Masked multi head attention

In [ ]:
class multiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False) -> None:
      super().__init__()
      assert (d_out % num_heads == 0), \
          "d_out must be divisible by num_heads"
      self.d_out = d_out
      self.num_heads = num_heads
      self.head_dim = d_out//num_heads # reduce the projection dim to match desire output dim
      self.query = nn.Linear(d_in,d_out,bias = qkv_bias)
      self.key = nn.Linear(d_in,d_out,bias = qkv_bias)
      self.value = nn.Linear(d_in,d_out,bias=qkv_bias)
      self.out_proj = nn.Linear(d_out,d_out) # linear layer to combine head op
      self.dropout = nn.Dropout(dropout)
      self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

  def forward(self,x):
    b, num_tokens, d_in = x.shape
    query = self.query(x) # b, num_tokens, d_out
    key = self.key(x)
    values = self.value(x)
    # we exlplictly split the matrix by adding a 'num_heads' dim
    # unroll last dim: b, num_tokens, d_out -> b, num_tokens, num_heads,head_dim
    keys = key.view(b,num_tokens,self.num_heads,self.head_dim)
    query = query.view(b,num_tokens,self.num_heads,self.head_dim)
    values = values.view(b,num_tokens,self.num_heads,self.head_dim)
    # transpose (b,num_tokens,num_heads,head_dim) -> (b,num_heads,num_tokens,head_dim)
    keys = keys.transpose(1,2)
    queries = query.transpose(1,2)
    values = values.transpose(1,2)

    # attention score
    att_score = queries@keys.transpose(2,3) # dot product for each head

    # Original mask truncated to the number of tokens and converted to boolean
    mask_bool = self.mask.bool()[:num_tokens,:num_tokens]
    # use the mask to fill attention score
    att_score.masked_fill(mask_bool, -torch.inf)

    # normalize the att score
    att_weights = torch.softmax(att_score/keys.shape[-1]**0.5,dim=-1)
    att_weights = self.dropout(att_weights)

    context_vec = (att_weights@values).transpose(1,2) # b,num_tokens,num_head,head_dim

    #Combine heads, where self.d_out = self.num_heads * self.head_dim
    context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
    context_vec = self.out_proj(context_vec) # optional projection
    return context_vec

Layer normalization

In [ ]:
class layerNorm(nn.Module):
  def __init__(self, emb_dim) -> None:
      super().__init__()
      self.eps = 1e-5
      self.scale = nn.Parameter(torch.ones(emb_dim))
      self.shift = nn.Parameter(torch.zeros(emb_dim))
  def forward(self,x):
    mean = x.mean(dim=-1,keepdim=True)
    var = x.var(dim=-1,keepdim=True)
    x_norm = (x-mean)/torch.sqrt(var+self.eps)
    return self.scale * x_norm + self.shift

Feed forward network with GELU activation function

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [ ]:
class feedForward(nn.Module):
  def __init__(self, cfg) -> None:
      super().__init__()
      self.layers = nn.Sequential(
          nn.Linear(cfg['emb_dim'],4*cfg['emb_dim']),
          GELU(),
          nn.Linear(4*cfg['emb_dim'],cfg['emb_dim'])
      )
  def forward(self,x):
    return self.layers(x)

Complete transformer block

In [ ]:
class transformerBlock(nn.Module):
  def __init__(self, cfg) -> None:
     super().__init__()
     self.att = multiHeadAttention(
         d_in = cfg['emb_dim'],
         d_out= cfg['emb_dim'],
         context_length=cfg['context_length'],
         dropout=cfg['drop_rate'],
         num_heads=cfg['n_heads'],
         qkv_bias= cfg['qkv_bias']
     )
     self.ff = feedForward(cfg)
     self.norm1 = layerNorm(cfg['emb_dim'])
     self.norm2 = layerNorm(cfg['emb_dim'])
     self.drop_shortcut = nn.Dropout(cfg['drop_rate'])
  def forward(self,x):
    # short cut connection for attention block
    shortcut = x
    x = self.norm1(x)
    x = self.att(x)
    x = self.drop_shortcut(x)
    x = x+shortcut

    # shortcut connection for the ff network
    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x = self.drop_shortcut(x)
    x = x+shortcut
    return x


5. Now lets code the entire GPT architecture

In [ ]:
class GPTModel(nn.Module):
  def __init__(self, cfg) -> None:
     super().__init__()
     self.tok_emb = nn.Embedding(cfg['vocab_size'],cfg['emb_dim'])
     self.pos_emb = nn.Embedding(cfg['context_length'],cfg['emb_dim'])
     self.drop_emb = nn.Dropout(cfg['drop_rate'])
     self.trf_blocks = nn.Sequential(
         *[transformerBlock(cfg) for _ in range(cfg['n_layers'])]
     )
     self.final_norm = layerNorm(cfg['emb_dim'])
     self.out_head = nn.Linear(cfg['emb_dim'],cfg['vocab_size'],bias = False)
  def forward(self,in_idx):
    b,seq_len = in_idx.shape
    tok_emb = self.tok_emb(in_idx)
    pos_emb = self.pos_emb(torch.arange(seq_len,device = in_idx.device))
    x = tok_emb + pos_emb # batch_size, num_tokens, emb_dim
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

6. Generating text from output token ids

<div class="alert alert-block alert-info">

Step 1: idx is a (batch, n_tokens) array of indices in the current context

Step 2: Crop current context if it exceeds the supported context size E.g., if LLM supports only 5 tokens, and the
context size is 10 then only the last 5 tokens are used as context

Step 3: Focus only on the last time step, so that (batch, n_token, vocab_size) becomes (batch, vocab_size)

Step 4: probas has shape (batch, vocab_size)

Step 5: idx_next has shape (batch, 1)

Step 6: Append sampled index to the running sequence, where idx has shape (batch, n_tokens+1)

</div>

In [ ]:
def generating_text_simple(model,idx,max_new_tokens,context_size):
  for _ in range(max_new_tokens):
    # Crop current context if it exceeds the supported context size
    # E.g., if LLM supports only 5 tokens, and the context size is 10
    # then only the last 5 tokens are used as context
    idx_count = idx[:,-context_size:]
    with torch.no_grad():
      logits = model(idx_count) # batch, num_tokens, vocab_size
    # Focus only on the last time step
    # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
    logits = logits[:, -1, :]
    # apply softmax
    probas = torch.softmax(logits,dim=-1)
    # get the idx of the vocab that have highest prob
    idx_next = torch.argmax(probas,dim=-1,keepdim=True)
    # Append sampled index to the running sequence
    idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

  return idx

7. Evaluating generative text models

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference

In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"



token_ids = generating_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you Aeiman flaws SPLabbcats paramed jur himself CBS


Calc training and validation loss

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Note:
# Uncommenting the following lines will allow the code to run on Apple Silicon chips, if applicable,
# which is approximately 2x faster than on an Apple CPU (as measured on an M3 MacBook Air).
# However, the resulting loss values may be slightly different.

#if torch.cuda.is_available():
#    device = torch.device("cuda")
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")
#else:
#    device = torch.device("cpu")
#
# print(f"Using {device} device.")


model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes


torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 10.999784469604492
Validation loss: nan


Training loop for LLM

In [ ]:
def train_simple_model(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
  train_losses,val_losses,train_token_seen = [],[],[]
  token_seen, global_step = 0,-1
  # main training loop
  for epoch in range(num_epochs):
    model.train()
    for input_batch,target_batch in train_loader:
      optimizer.zero_grad() # reset loss gradients from previous batch iteration
      loss = calc_loss_batch(input_batch,target_batch)